# 🧪 Model Evaluation Dashboard

This notebook compares four model variants across three evaluation frameworks:
- **Comprehensive Evaluation** — syntax validity, CodeBLEU, token efficiency, etc.
- **LLM-as-a-Judge** — correctness, code quality, best practices, readability
- **Pass@k** — functional correctness via unit tests

**Models compared:**
| Alias | Description | Device |
|---|---|---|
| `base_model` | Qwen2.5-Coder-0.5B-Instruct (no fine-tuning) | GPU |
| `fine_tuned` | Fine-tuned on domain data | GPU |
| `onnx` | Fine-tuned → ONNX export | CPU |
| `onnx_quantized` | Fine-tuned → ONNX + INT8 quantization | CPU |

In [1]:
# ── Imports ────────────────────────────────────────────────────────────────
import json
import os
from pathlib import Path
import sys

import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

# Output directory for saved plots
PLOTS_DIR = ROOT_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

print("✅ Imports OK — plots will be saved to:", PLOTS_DIR.resolve())

✅ Imports OK — plots will be saved to: /project_resources/private_workspace/krushna/qwen/plots


In [2]:
# ── File Paths  ────────────────────────────────────────────────────────────
# TODO: Update these paths to match your actual file locations

PATHS = {
    "comprehensive": {
        "base_model":      f"{str(ROOT_DIR)}/evaluation_advanced/results/comprehensive_eval/base_model.json",
        "fine_tuned":      f"{str(ROOT_DIR)}/evaluation_advanced/results/comprehensive_eval/fine_tuned.json",
        "onnx":            f"{str(ROOT_DIR)}/evaluation_advanced/results/comprehensive_eval/onnx_model.json",
        "onnx_quantized":  f"{str(ROOT_DIR)}/evaluation_advanced/results/comprehensive_eval/onnx_quantized.json",
    },
    "llm_judge": {
        "base_model":      f"{str(ROOT_DIR)}/evaluation_advanced/results/llm_judge/base_model.json",
        "fine_tuned":      f"{str(ROOT_DIR)}/evaluation_advanced/results/llm_judge/fine_tuned.json",
        "onnx":            f"{str(ROOT_DIR)}/evaluation_advanced/results/llm_judge/onnx_model.json",
        "onnx_quantized":  f"{str(ROOT_DIR)}/evaluation_advanced/results/llm_judge/onnx_quantized.json",
    },
    "pass_at_k": {
        "base_model":      f"{str(ROOT_DIR)}/evaluation_advanced/results/pass_at_k/base_model.json",
        "fine_tuned":      f"{str(ROOT_DIR)}/evaluation_advanced/results/pass_at_k/fine_tuned.json",
        "onnx":            f"{str(ROOT_DIR)}/evaluation_advanced/results/pass_at_k/onnx_model.json",
        "onnx_quantized":  f"{str(ROOT_DIR)}/evaluation_advanced/results/pass_at_k/onnx_quantized.json",
    },
}

print("📂 Path config loaded")

📂 Path config loaded


In [3]:
# ── Data Loading ───────────────────────────────────────────────────────────

def load_json(path: str) -> dict:
    with open(path, "r") as f:
        return json.load(f)

def load_all(paths: dict) -> dict:
    return {name: load_json(p) for name, p in paths.items()}

comprehensive = load_all(PATHS["comprehensive"])
llm_judge     = load_all(PATHS["llm_judge"])
pass_at_k     = load_all(PATHS["pass_at_k"])

print("✅ All 12 JSON files loaded successfully")

# Friendly display names and colors
MODEL_LABELS = {
    "base_model":     "Base Model",
    "fine_tuned":     "Fine-Tuned",
    "onnx":           "ONNX (CPU)",
    "onnx_quantized": "ONNX Quantized (CPU)",
}

COLORS = {
    "base_model":     "#5B8CFF",
    "fine_tuned":     "#2ECC71",
    "onnx":           "#F39C12",
    "onnx_quantized": "#E74C3C",
}

MODELS = list(MODEL_LABELS.keys())

def m(key):
    """Extract metrics dict from comprehensive eval."""
    return comprehensive[key]["metrics"]

def save_plot(fig, filename: str):
    """Save plot as interactive HTML and static PNG."""
    html_path = PLOTS_DIR / f"{filename}.html"
    png_path  = PLOTS_DIR / f"{filename}.png"
    fig.write_html(str(html_path))
    try:
        fig.write_image(str(png_path), scale=2)
        print(f"  💾 Saved → {html_path}  &  {png_path}")
    except Exception:
        print(f"  💾 Saved → {html_path}  (install kaleido for PNG: pip install kaleido)")

✅ All 12 JSON files loaded successfully


---
## 📊 Plot 1 — Core Quality Metrics Overview

Grouped bar chart comparing **Syntax Validity**, **Compilation Rate**, **CodeBLEU**, and **Exact Match** across all four models.
These metrics capture how syntactically and semantically correct each model's output is.

In [4]:
# Plot 1 — Core quality metrics grouped bar chart

metrics_cfg = [
    ("syntax_validity_pct",  "Syntax Validity %"),
    ("compilation_rate_pct", "Compilation Rate %"),
    ("exact_match_pct",      "Exact Match %"),
]

fig1 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[label for _, label in metrics_cfg],
    shared_yaxes=False,
)

for col_idx, (key, label) in enumerate(metrics_cfg, start=1):
    for model in MODELS:
        val = m(model)[key]
        fig1.add_trace(
            go.Bar(
                name=MODEL_LABELS[model],
                x=[MODEL_LABELS[model]],
                y=[val],
                marker_color=COLORS[model],
                showlegend=(col_idx == 1),
                hovertemplate=f"<b>{MODEL_LABELS[model]}</b><br>{label}: %{{y:.2f}}<extra></extra>",
            ),
            row=1, col=col_idx,
        )

fig1.update_layout(
    title_text="Core Quality Metrics — All Models",
    title_font_size=18,
    barmode="group",
    height=450,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    margin=dict(t=80, b=100),
)

fig1.show()
save_plot(fig1, "01_core_quality_metrics")

  💾 Saved → /project_resources/private_workspace/krushna/qwen/plots/01_core_quality_metrics.html  &  /project_resources/private_workspace/krushna/qwen/plots/01_core_quality_metrics.png


---
## 📊 Plot 2 — CodeBLEU Score Comparison

**CodeBLEU** is a code-specific similarity metric combining token match, AST match, and data-flow match against reference solutions.
Higher = better semantic alignment with ground truth.

In [5]:
# Plot 2 — CodeBLEU horizontal bar chart with improvement annotations

labels = [MODEL_LABELS[m_] for m_ in MODELS]
values = [m(m_)["codebleu"] for m_ in MODELS]
colors = [COLORS[m_] for m_ in MODELS]

base_val = values[0]
annotations = []
for i, (v, model) in enumerate(zip(values, MODELS)):
    delta = ((v - base_val) / base_val) * 100 if model != "base_model" else 0
    sign = "+" if delta >= 0 else ""
    tag = f"  Baseline" if model == "base_model" else f"  {sign}{delta:.1f}% vs base"
    annotations.append(dict(
        x=v, y=i,
        text=tag,
        showarrow=False,
        xanchor="left",
        font=dict(size=12, color="#444"),
    ))

fig2 = go.Figure(go.Bar(
    x=values,
    y=labels,
    orientation="h",
    marker_color=colors,
    hovertemplate="<b>%{y}</b><br>CodeBLEU: %{x:.4f}<extra></extra>",
    text=[f"{v:.4f}" for v in values],
    textposition="inside",
    textfont=dict(color="white", size=13),
))

fig2.update_layout(
    title_text="CodeBLEU Score Comparison",
    title_font_size=18,
    xaxis_title="CodeBLEU Score (higher = better)",
    template="plotly_white",
    height=350,
    annotations=annotations,
    xaxis=dict(range=[0, max(values) * 1.35]),
    margin=dict(t=60, r=20),
)

fig2.show()
save_plot(fig2, "02_codebleu_comparison")

  💾 Saved → /project_resources/private_workspace/krushna/qwen/plots/02_codebleu_comparison.html  &  /project_resources/private_workspace/krushna/qwen/plots/02_codebleu_comparison.png


---
## 📊 Plot 3 — Token Efficiency & Response Verbosity

Compares **average tokens**, **average response length (chars)**, and **excess explanation %**.
The fine-tuned and ONNX models should be significantly more concise than the base model, which tends to over-explain.

In [6]:
# Plot 3 — Token efficiency: avg_tokens + excess_explanation_pct

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Average Tokens per Response", "Excess Explanation %"],
    specs=[[{"type": "bar"}, {"type": "bar"}]],
)

labels = [MODEL_LABELS[m_] for m_ in MODELS]
avg_tokens = [m(m_)["avg_tokens"] for m_ in MODELS]
excess_pct  = [m(m_)["excess_explanation_pct"] for m_ in MODELS]
colors_list = [COLORS[m_] for m_ in MODELS]

fig3.add_trace(go.Bar(
    x=labels, y=avg_tokens,
    marker_color=colors_list,
    hovertemplate="<b>%{x}</b><br>Avg Tokens: %{y:.1f}<extra></extra>",
    showlegend=False,
), row=1, col=1)

fig3.add_trace(go.Bar(
    x=labels, y=excess_pct,
    marker_color=colors_list,
    hovertemplate="<b>%{x}</b><br>Excess Explanation: %{y:.1f}%<extra></extra>",
    showlegend=False,
), row=1, col=2)

fig3.update_yaxes(title_text="Tokens", row=1, col=1)
fig3.update_yaxes(title_text="% Samples with Excess Explanation", row=1, col=2)

fig3.update_layout(
    title_text="Token Efficiency & Verbosity (lower = more concise)",
    title_font_size=18,
    template="plotly_white",
    height=420,
    margin=dict(t=80, b=60),
)

fig3.show()
save_plot(fig3, "03_token_efficiency")

  💾 Saved → /project_resources/private_workspace/krushna/qwen/plots/03_token_efficiency.html  &  /project_resources/private_workspace/krushna/qwen/plots/03_token_efficiency.png


---
## 📊 Plot 4 — Code Ratio Analysis

**Code ratio** = fraction of the response that is actual code (vs prose/explanation).
A ratio near 1.0 means the model outputs clean code with minimal surrounding text — desirable for code generation tasks.

In [7]:
# Plot 4 — Code ratio with token distribution (avg ± std)

labels = [MODEL_LABELS[m_] for m_ in MODELS]
code_ratio  = [m(m_)["avg_code_ratio"] for m_ in MODELS]
avg_tok     = [m(m_)["avg_tokens"] for m_ in MODELS]
med_tok     = [m(m_)["median_tokens"] for m_ in MODELS]
std_tok     = [m(m_)["std_tokens"] for m_ in MODELS]
colors_list = [COLORS[m_] for m_ in MODELS]

fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Avg Code Ratio (higher = more code, less prose)",
                    "Token Distribution (mean ± std)"],
)

fig4.add_trace(go.Bar(
    x=labels, y=code_ratio,
    marker_color=colors_list,
    text=[f"{v:.3f}" for v in code_ratio],
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Code Ratio: %{y:.3f}<extra></extra>",
    showlegend=False,
), row=1, col=1)
fig4.update_yaxes(range=[0, 1.15], row=1, col=1)

fig4.add_trace(go.Bar(
    x=labels, y=avg_tok,
    error_y=dict(type="data", array=std_tok, visible=True, color="#555"),
    marker_color=colors_list,
    hovertemplate="<b>%{x}</b><br>Mean: %{y:.1f} tokens<br>Median: " +
                  str(med_tok) + "<extra></extra>",
    showlegend=False,
), row=1, col=2)

# Cleaner hover for token distribution
fig4.data[-1].customdata = list(zip(med_tok, std_tok))
fig4.data[-1].hovertemplate = (
    "<b>%{x}</b><br>Mean: %{y:.1f}<br>"
    "Median: %{customdata[0]:.0f}<br>Std: %{customdata[1]:.1f}<extra></extra>"
)

fig4.update_yaxes(title_text="Code Ratio", row=1, col=1)
fig4.update_yaxes(title_text="Tokens", row=1, col=2)

fig4.update_layout(
    title_text="Code Ratio & Token Distribution",
    title_font_size=18,
    template="plotly_white",
    height=420,
    margin=dict(t=80, b=60),
)

fig4.show()
save_plot(fig4, "04_code_ratio_and_tokens")

  💾 Saved → /project_resources/private_workspace/krushna/qwen/plots/04_code_ratio_and_tokens.html  &  /project_resources/private_workspace/krushna/qwen/plots/04_code_ratio_and_tokens.png


---
## 📊 Plot 5 — Inference Speed Comparison

**Average time per sample (seconds)** measured during evaluation.
GPU models (base, fine-tuned) are faster; CPU ONNX models trade speed for portability — quantization adds overhead here surprisingly.

In [8]:
# Plot 5 — Inference speed: avg time per sample

labels    = [MODEL_LABELS[m_] for m_ in MODELS]
times     = [m(m_)["avg_time_per_sample"] for m_ in MODELS]
gen_total = [m(m_)["generation_time_sec"] for m_ in MODELS]
devices   = [comprehensive[m_]["metadata"].get("device", "?") for m_ in MODELS]
colors_list = [COLORS[m_] for m_ in MODELS]

fig5 = go.Figure()

fig5.add_trace(go.Bar(
    x=labels,
    y=times,
    marker_color=colors_list,
    customdata=list(zip(gen_total, devices)),
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Avg time/sample: %{y:.3f}s<br>"
        "Total gen time: %{customdata[0]:.1f}s<br>"
        "Device: %{customdata[1]}<extra></extra>"
    ),
    text=[f"{t:.2f}s" for t in times],
    textposition="outside",
))

# Add device annotation
for i, (label, device) in enumerate(zip(labels, devices)):
    fig5.add_annotation(
        x=label, y=0,
        text=f"[{device}]",
        showarrow=False, yanchor="top",
        yshift=-25, font=dict(size=10, color="#888"),
    )

fig5.update_layout(
    title_text="Inference Speed — Avg Time per Sample (lower = faster)",
    title_font_size=18,
    yaxis_title="Seconds per Sample",
    template="plotly_white",
    height=420,
    showlegend=False,
    margin=dict(t=70, b=80),
)

fig5.show()
save_plot(fig5, "05_inference_speed")

  💾 Saved → /project_resources/private_workspace/krushna/qwen/plots/05_inference_speed.html  &  /project_resources/private_workspace/krushna/qwen/plots/05_inference_speed.png


---
## 📊 Plot 6 — LLM-as-a-Judge Scores

A separate Claude-based judge evaluated 100 samples per model on four axes:
- **Correctness** — does the code do what was asked?
- **Code Quality** — is it well-structured?
- **Best Practices** — follows language idioms?
- **Readability** — naming, comments, clarity?

Scores are out of 10.

In [9]:
# Plot 6 — LLM-as-a-Judge grouped bar chart

judge_metrics = ["correctness", "code_quality", "best_practices", "readability", "overall"]
judge_labels  = ["Correctness", "Code Quality", "Best Practices", "Readability", "Overall"]

fig6 = go.Figure()

for model in MODELS:
    scores = llm_judge[model]["aggregate_scores"]
    vals   = [scores[k] for k in judge_metrics]
    fig6.add_trace(go.Bar(
        name=MODEL_LABELS[model],
        x=judge_labels,
        y=vals,
        marker_color=COLORS[model],
        hovertemplate="<b>" + MODEL_LABELS[model] + "</b><br>%{x}: %{y:.2f}/10<extra></extra>",
    ))

fig6.update_layout(
    title_text="LLM-as-a-Judge Scores (out of 10)",
    title_font_size=18,
    barmode="group",
    yaxis=dict(title="Score / 10", range=[0, 10]),
    template="plotly_white",
    height=460,
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    margin=dict(t=70, b=100),
)

fig6.show()
save_plot(fig6, "06_llm_judge_scores")

  💾 Saved → /project_resources/private_workspace/krushna/qwen/plots/06_llm_judge_scores.html  &  /project_resources/private_workspace/krushna/qwen/plots/06_llm_judge_scores.png


---
## 📊 Plot 7 — LLM Judge: Radar Chart per Model

Radar (spider) chart showing the LLM judge score profile for each model.
Helps visualise where each model is strong or weak at a glance.

In [10]:
# Plot 7 — Radar chart of LLM judge scores

radar_keys   = ["correctness", "code_quality", "best_practices", "readability"]
radar_labels = ["Correctness", "Code Quality", "Best Practices", "Readability"]
# Close the polygon
radar_labels_closed = radar_labels + [radar_labels[0]]

fig7 = go.Figure()

for model in MODELS:
    scores = llm_judge[model]["aggregate_scores"]
    vals   = [scores[k] for k in radar_keys]
    vals_closed = vals + [vals[0]]
    fig7.add_trace(go.Scatterpolar(
        r=vals_closed,
        theta=radar_labels_closed,
        name=MODEL_LABELS[model],
        line=dict(color=COLORS[model], width=2),
        fill="toself",
        fillcolor=COLORS[model],
        opacity=0.18,
        hovertemplate="<b>" + MODEL_LABELS[model] + "</b><br>%{theta}: %{r:.2f}<extra></extra>",
    ))

fig7.update_layout(
    title_text="LLM Judge — Score Profile Radar",
    title_font_size=18,
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 10], tickfont_size=9),
    ),
    template="plotly_white",
    height=520,
    legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5),
)

fig7.show()
save_plot(fig7, "07_llm_judge_radar")

  💾 Saved → /project_resources/private_workspace/krushna/qwen/plots/07_llm_judge_radar.html  &  /project_resources/private_workspace/krushna/qwen/plots/07_llm_judge_radar.png


---
## 📊 Plot 8 — Pass@k Results

**Pass@k** measures functional correctness: given k generated solutions per problem, what fraction of problems has at least one passing solution?
- **Pass@1** — single-shot accuracy
- **Pass@3** — best of 3
- **Pass@5** — best of 5

Evaluated on 50 problems with unit tests.

In [11]:
# Plot 8 — Pass@k grouped bar chart

k_values = ["pass@1", "pass@3", "pass@5"]
k_labels = ["Pass@1", "Pass@3", "Pass@5"]

fig8 = go.Figure()

for model in MODELS:
    pak = pass_at_k[model]["pass_at_k"]
    vals = [pak[k] for k in k_values]
    total_passed = pass_at_k[model]["metadata"]["total_solutions_passed"]
    fig8.add_trace(go.Bar(
        name=MODEL_LABELS[model],
        x=k_labels,
        y=vals,
        marker_color=COLORS[model],
        customdata=[[total_passed]*3],
        hovertemplate="<b>" + MODEL_LABELS[model] + "</b><br>%{x}: %{y:.1f}%<extra></extra>",
    ))

fig8.update_layout(
    title_text="Pass@k — Functional Correctness (%, higher = better)",
    title_font_size=18,
    barmode="group",
    yaxis=dict(title="Pass Rate (%)", range=[0, 100]),
    template="plotly_white",
    height=460,
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    margin=dict(t=70, b=100),
)

fig8.show()
save_plot(fig8, "08_pass_at_k")

  💾 Saved → /project_resources/private_workspace/krushna/qwen/plots/08_pass_at_k.html  &  /project_resources/private_workspace/krushna/qwen/plots/08_pass_at_k.png


---
## 📊 Plot 9 — Pass@k: Improvement Curves

Line chart showing how pass rate grows as k increases.
Steeper curves indicate the model generates diverse solutions — one of them is likely to be correct even if others aren't.

In [12]:
# Plot 9 — Pass@k line curves (diversity of solutions)

k_x = [1, 3, 5]
k_keys = ["pass@1", "pass@3", "pass@5"]

fig9 = go.Figure()

for model in MODELS:
    pak  = pass_at_k[model]["pass_at_k"]
    vals = [pak[k] for k in k_keys]
    fig9.add_trace(go.Scatter(
        x=k_x, y=vals,
        mode="lines+markers",
        name=MODEL_LABELS[model],
        line=dict(color=COLORS[model], width=2.5),
        marker=dict(size=9, color=COLORS[model]),
        hovertemplate="<b>" + MODEL_LABELS[model] + "</b><br>k=%{x}<br>Pass Rate: %{y:.1f}%<extra></extra>",
    ))

fig9.update_layout(
    title_text="Pass@k Curves — Solution Diversity",
    title_font_size=18,
    xaxis=dict(title="k (number of attempts)", tickvals=k_x),
    yaxis=dict(title="Pass Rate (%)", range=[0, 95]),
    template="plotly_white",
    height=440,
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    margin=dict(t=70, b=100),
)

fig9.show()
save_plot(fig9, "09_pass_at_k_curves")

  💾 Saved → /project_resources/private_workspace/krushna/qwen/plots/09_pass_at_k_curves.html  &  /project_resources/private_workspace/krushna/qwen/plots/09_pass_at_k_curves.png


---
## 📊 Plot 10 — Comprehensive Radar: All Normalised Metrics

All metrics from the comprehensive evaluation normalised to [0, 1] and plotted on a single radar.
Lower-is-better metrics (excess explanation, avg tokens, time) are **inverted** so that outward = better for all axes.

In [13]:
# Plot 10 — Comprehensive normalised radar (all metrics)

radar_cfg = [
    ("syntax_validity_pct",   "Syntax\nValidity",    True),
    ("codebleu",              "CodeBLEU",             True),
    ("exact_match_pct",       "Exact\nMatch",         True),
    ("avg_code_ratio",        "Code Ratio",           True),
    ("excess_explanation_pct","Low\nExcess",          False),   # inverted
    ("avg_tokens",            "Concise\nTokens",      False),   # inverted
    ("avg_time_per_sample",   "Speed",                False),   # inverted
]

def norm(v, mn, mx): return (v - mn) / (mx - mn) if mx != mn else 0.5
def inv_norm(v, mn, mx): return 1 - norm(v, mn, mx)

r_labels = [cfg[1] for cfg in radar_cfg] + [radar_cfg[0][1]]

fig10 = go.Figure()

for model in MODELS:
    vals = []
    for key, _, hib in radar_cfg:
        all_v = [m(m_)[key] for m_ in MODELS]
        mn, mx = min(all_v), max(all_v)
        v = m(model)[key]
        vals.append(norm(v, mn, mx) if hib else inv_norm(v, mn, mx))
    vals_closed = vals + [vals[0]]
    fig10.add_trace(go.Scatterpolar(
        r=vals_closed,
        theta=r_labels,
        name=MODEL_LABELS[model],
        line=dict(color=COLORS[model], width=2),
        fill="toself",
        fillcolor=COLORS[model],
        opacity=0.18,
        hovertemplate="<b>" + MODEL_LABELS[model] + "</b><br>%{theta}: %{r:.2f}<extra></extra>",
    ))

fig10.update_layout(
    title_text="Comprehensive Normalised Radar (outward = better)",
    title_font_size=18,
    polar=dict(radialaxis=dict(visible=True, range=[0, 1], tickfont_size=8)),
    template="plotly_white",
    height=540,
    legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5),
)

fig10.show()
save_plot(fig10, "10_comprehensive_radar")

  💾 Saved → /project_resources/private_workspace/krushna/qwen/plots/10_comprehensive_radar.html  &  /project_resources/private_workspace/krushna/qwen/plots/10_comprehensive_radar.png


---
## 📊 Plot 11 — Composite Leaderboard

A weighted composite score combining:
- Syntax validity (25%), CodeBLEU (25%), Code ratio (15%) from comprehensive eval
- LLM Judge overall score (20%)
- Pass@1 (15%)

All sub-scores normalised to [0, 1] before weighting. This gives a single ranking number.

In [14]:
# Plot 11 — Composite leaderboard score

COMPOSITE_WEIGHTS = {
    "syntax_validity_pct":  0.20,
    "codebleu":             0.20,
    "avg_code_ratio":       0.10,
    "excess_explanation_pct": 0.05,  # inverted
    "avg_tokens":           0.05,    # inverted
    "avg_time_per_sample":  0.05,    # inverted
    "llm_judge_overall":    0.20,
    "pass_at_1":            0.15,
}

# Build combined data per model
combined = {}
for model in MODELS:
    combined[model] = {
        **comprehensive[model]["metrics"],
        "llm_judge_overall": llm_judge[model]["aggregate_scores"]["overall"],
        "pass_at_1": pass_at_k[model]["pass_at_k"]["pass@1"],
    }

# Normalise & compute
hib_map = {
    "syntax_validity_pct": True, "codebleu": True, "avg_code_ratio": True,
    "excess_explanation_pct": False, "avg_tokens": False, "avg_time_per_sample": False,
    "llm_judge_overall": True, "pass_at_1": True,
}

def composite(model_key):
    score = 0.0
    for k, w in COMPOSITE_WEIGHTS.items():
        all_v = [combined[m_][k] for m_ in MODELS]
        mn, mx = min(all_v), max(all_v)
        v = combined[model_key][k]
        n = norm(v, mn, mx) if hib_map[k] else inv_norm(v, mn, mx)
        score += w * n
    return score

scores    = {model: composite(model) for model in MODELS}
ranked    = sorted(scores.items(), key=lambda x: -x[1])
rank_labels = [MODEL_LABELS[m_] for m_, _ in ranked]
rank_scores = [s for _, s in ranked]
rank_colors = [COLORS[m_] for m_, _ in ranked]

fig11 = go.Figure(go.Bar(
    x=rank_labels,
    y=rank_scores,
    marker_color=rank_colors,
    text=[f"{s:.3f}" for s in rank_scores],
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Composite Score: %{y:.3f}<extra></extra>",
))

for i, (label, score) in enumerate(zip(rank_labels, rank_scores)):
    medal = ["🥇", "🥈", "🥉", "4th"][i]
    fig11.add_annotation(
        x=label, y=score + 0.015,
        text=medal, showarrow=False,
        font=dict(size=20),
    )

fig11.update_layout(
    title_text="Composite Leaderboard Score (higher = better)",
    title_font_size=18,
    yaxis=dict(title="Composite Score", range=[0, 1.1]),
    template="plotly_white",
    height=440,
    showlegend=False,
    margin=dict(t=70, b=60),
)

fig11.show()
save_plot(fig11, "11_composite_leaderboard")

print("\n🏆 Final Ranking:")
for i, (model, score) in enumerate(ranked, 1):
    print(f"  {i}. {MODEL_LABELS[model]:<25} {score:.3f}")

  💾 Saved → /project_resources/private_workspace/krushna/qwen/plots/11_composite_leaderboard.html  &  /project_resources/private_workspace/krushna/qwen/plots/11_composite_leaderboard.png

🏆 Final Ranking:
  1. Fine-Tuned                0.971
  2. ONNX (CPU)                0.889
  3. Base Model                0.371
  4. ONNX Quantized (CPU)      0.263


---
## 📊 Plot 12 — Quality vs Speed Trade-off

Scatter plot: **x-axis = inference time** (seconds per sample), **y-axis = composite quality score**.
Bubble size = avg tokens. The ideal model sits top-left (fast + high quality).

In [15]:
# Plot 12 — Quality vs Speed trade-off bubble chart

times_all  = [m(model)["avg_time_per_sample"] for model in MODELS]
scores_all = [scores[model] for model in MODELS]
tokens_all = [m(model)["avg_tokens"] for model in MODELS]
labels_all = [MODEL_LABELS[model] for model in MODELS]
colors_all = [COLORS[model] for model in MODELS]
devices_all = [comprehensive[model]["metadata"].get("device", "?") for model in MODELS]

fig12 = go.Figure()

for i, model in enumerate(MODELS):
    fig12.add_trace(go.Scatter(
        x=[times_all[i]],
        y=[scores_all[i]],
        mode="markers+text",
        name=labels_all[i],
        marker=dict(
            size=tokens_all[i] / 6,
            color=colors_all[i],
            opacity=0.85,
            line=dict(width=2, color="white"),
        ),
        text=[labels_all[i]],
        textposition="top center",
        customdata=[[tokens_all[i], devices_all[i]]],
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Time/sample: %{x:.3f}s<br>"
            "Composite score: %{y:.3f}<br>"
            "Avg tokens: %{customdata[0]:.0f}<br>"
            "Device: %{customdata[1]}<extra></extra>"
        ),
    ))

# Annotate ideal zone
fig12.add_annotation(
    x=min(times_all) * 0.95, y=max(scores_all) * 1.02,
    text="← Ideal zone", showarrow=False,
    font=dict(size=11, color="gray"), xanchor="left",
)

fig12.update_layout(
    title_text="Quality vs Speed Trade-off  (bubble size ∝ avg tokens)",
    title_font_size=18,
    xaxis_title="Avg Inference Time per Sample (s)  — lower = faster",
    yaxis_title="Composite Quality Score  — higher = better",
    template="plotly_white",
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5),
    margin=dict(t=70, b=100),
)

fig12.show()
save_plot(fig12, "12_quality_vs_speed")

  💾 Saved → /project_resources/private_workspace/krushna/qwen/plots/12_quality_vs_speed.html  &  /project_resources/private_workspace/krushna/qwen/plots/12_quality_vs_speed.png


---
## ✅ Summary

All plots generated and saved to the `plots/` directory.

| # | Plot | File |
|---|------|------|
| 1 | Core quality metrics (syntax, compilation, exact match) | `01_core_quality_metrics` |
| 2 | CodeBLEU score comparison | `02_codebleu_comparison` |
| 3 | Token efficiency & verbosity | `03_token_efficiency` |
| 4 | Code ratio & token distribution | `04_code_ratio_and_tokens` |
| 5 | Inference speed | `05_inference_speed` |
| 6 | LLM-as-a-Judge scores | `06_llm_judge_scores` |
| 7 | LLM Judge radar chart | `07_llm_judge_radar` |
| 8 | Pass@k bar chart | `08_pass_at_k` |
| 9 | Pass@k improvement curves | `09_pass_at_k_curves` |
| 10 | Comprehensive normalised radar | `10_comprehensive_radar` |
| 11 | Composite leaderboard | `11_composite_leaderboard` |
| 12 | Quality vs Speed trade-off | `12_quality_vs_speed` |

Each file exists as both `.html` (interactive, for browser) and `.png` (static, for reports).

In [16]:
# Final check — list all saved plot files
print("📁 Saved plots:")
for f in sorted(PLOTS_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45} {size_kb:>7.1f} KB")

📁 Saved plots:
  01_core_quality_metrics.html                   4745.6 KB
  01_core_quality_metrics.png                     117.3 KB
  02_codebleu_comparison.html                    4742.3 KB
  02_codebleu_comparison.png                       84.2 KB
  03_token_efficiency.html                       4742.5 KB
  03_token_efficiency.png                         125.3 KB
  04_code_ratio_and_tokens.html                  4743.0 KB
  04_code_ratio_and_tokens.png                    126.3 KB
  05_inference_speed.html                        4742.5 KB
  05_inference_speed.png                           75.6 KB
  06_llm_judge_scores.html                       4742.7 KB
  06_llm_judge_scores.png                          80.0 KB
  07_llm_judge_radar.html                        4742.9 KB
  07_llm_judge_radar.png                          141.1 KB
  08_pass_at_k.html                              4742.6 KB
  08_pass_at_k.png                                 82.2 KB
  09_pass_at_k_curves.html               